# VibeThinker Local Chat — Colab Demo Server

Run the full **VibeThinker Local Chat** (FastAPI server + Web UI) in Google Colab,
with a public tunnel for taking screenshots and screen recordings.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OMCHOKSI108/VibeThinkerModel/blob/main/notebooks/VibeThinker_Local_Chat_Server_Colab_Demo.ipynb)

**Attribution**

| Resource | Link |
|----------|------|
| Original Model | [WeiboAI/VibeThinker-3B](https://huggingface.co/WeiboAI/VibeThinker-3B) |
| Original GitHub | [WeiboAI/VibeThinker](https://github.com/WeiboAI/VibeThinker) |
| This Fork | [OMCHOKSI108/VibeThinkerModel](https://github.com/OMCHOKSI108/VibeThinkerModel) |
| Technical Report | [arXiv:2606.16140](https://arxiv.org/pdf/2606.16140) |

> **Runtime**: `Runtime` → `Change runtime type` → `T4 GPU` (16 GB VRAM)
> The model requires ~8 GB VRAM in bfloat16 or ~3.5 GB in 4-bit mode.

## 1. Runtime Setup

Verify GPU availability.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected. Switch to a GPU runtime.")

## 2. Clone Repository & Install Dependencies

Clones the VibeThinkerModel repo and installs the server Python dependencies.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/OMCHOKSI108/VibeThinkerModel.git"
NOTEBOOK_DIR = "/content/VibeThinkerModel"

if not os.path.exists(NOTEBOOK_DIR):
    !git clone --depth 1 {REPO_URL}
else:
    print("Repo already cloned. Pulling latest...")
    %cd {NOTEBOOK_DIR}
    !git pull

# Navigate to server directory
%cd {NOTEBOOK_DIR}/server

print(f"\nWorking directory: {os.getcwd()}")
print("\nInstalling Python packages...")

# Install server dependencies (relaxed pins for Colab compatibility)
!pip install -q \
    "fastapi>=0.100.0" \
    "uvicorn[standard]>=0.20.0" \
    "strawberry-graphql[fastapi]>=0.200.0" \
    "transformers>=4.37.0" \
    "torch>=2.0.0" \
    "accelerate>=0.20.0" \
    "bitsandbytes>=0.40.0" \
    "sentencepiece>=0.1.99" \
    "python-dotenv>=1.0.0" \
    "pydantic>=2.0.0"

print("\n✅ Dependencies installed.")

## 3. Start the FastAPI Server

Launches the VibeThinker chat server in the background. The model will be downloaded
from Hugging Face on first run (2-5 minutes).

In [ ]:
import os, subprocess, time, sys, urllib.request, json

# Environment for Colab
os.environ["HOST"] = "0.0.0.0"
os.environ["PORT"] = "8000"
os.environ["LOAD_IN_4BIT"] = "true"
os.environ["MODEL_ID"] = "OMCHOKSI108/VibeThinker-3B"
os.environ["MAX_NEW_TOKENS"] = "4096"
os.environ["TEMPERATURE"] = "0.6"
os.environ["TOP_P"] = "0.95"

# Kill leftover processes
!pkill -f "run.py" 2>/dev/null || true
!pkill -f "uvicorn" 2>/dev/null || true
time.sleep(1)

# Start server in background
log_file = open("server.log", "w", buffering=1)
server_proc = subprocess.Popen(
    [sys.executable, "run.py"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"Server PID: {server_proc.pid}")
print("Loading model (first download may take 2-5 min)...\n")

# Poll /health until ready
start = time.time()
timeout = 600  # 10 minutes
ready = False

while time.time() - start < timeout:
    elapsed = int(time.time() - start)
    try:
        req = urllib.request.Request("http://localhost:8000/health")
        with urllib.request.urlopen(req, timeout=5) as resp:
            data = json.loads(resp.read())
            if data.get("model_loaded"):
                print(f"  ✅ Model ready on {data['device']} ({elapsed}s)")
                ready = True
                break
            elif data.get("load_error"):
                print(f"  ❌ Model failed: {data['load_error']}")
                break
            else:
                if elapsed % 15 == 0:
                    print(f"  Model still loading... ({elapsed}s)")
    except Exception:
        if elapsed % 15 == 0:
            print(f"  Waiting for server to start... ({elapsed}s)")
    time.sleep(5)
else:
    print("  ⚠️ Timeout waiting for server!")

print()
if ready:
    print("🎯 Server is running at: http://localhost:8000")
    print("💬 Chat client at:       http://localhost:8000/client")
else:
    print("Server may not be fully ready. Check logs below.")
    !tail -20 server.log

## 4. Expose with Cloudflare Tunnel

Creates a public URL so you can access the chat UI from your browser
for screenshots and recordings.

> **Note**: Cloudflare Tunnel is free and requires no signup.

In [ ]:
import subprocess, time, re, os, urllib.request

CLOUDFLARED = "./cloudflared"

# Download cloudflared if not present
if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    !wget -q "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64" -O {CLOUDFLARED}
    !chmod +x {CLOUDFLARED}
    print("Done.\n")
else:
    print("cloudflared already downloaded.\n")

# Start tunnel
tunnel_log = open("tunnel.log", "w", buffering=1)
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for tunnel URL
print("Starting tunnel...", end="", flush=True)
time.sleep(8)
print()

# Read and parse log
tunnel_log.flush()
with open("tunnel.log") as f:
    output = f.read()

match = re.search(r'https://[a-zA-Z0-9_-]+\.trycloudflare\.com', output)
if match:
    public_url = match.group(0)
    print(f"\n{'='*60}")
    print(f"  🌐  Public URL:        {public_url}")
    print(f"  💬  Chat Client:       {public_url}/client")
    print(f"  🔍  Health Check:      {public_url}/health")
    print(f"{'='*60}")
    print()
    print("Open the Chat Client URL in your browser to use the UI.")
    print("Take screenshots or record your screen as needed.")
else:
    print("Could not find tunnel URL. Showing last lines of log:")
    !tail -20 tunnel.log
    print()
    print("📌 Alternative: Use the ngrok method (next cell)")
    print("   Or access locally at http://localhost:8000/client")
    print("   (local access only works if browser is on the same machine)")

## (Alternative) Expose with ngrok

If Cloudflare Tunnel doesn't work, use ngrok instead.
Requires a **free ngrok account** and auth token from https://dashboard.ngrok.com/auth.

In [ ]:
# Only run this if cloudflared above didn't work
import subprocess, time

NGROK_TOKEN = ""  # <-- Paste your ngrok auth token here

if not NGROK_TOKEN:
    print("Please get a free auth token from https://dashboard.ngrok.com/auth")
    print("and paste it in the NGROK_TOKEN variable above.")
else:
    !pip install -q pyngrok

    from pyngrok import ngrok, conf

    # Configure and connect
    !ngrok authtoken {NGROK_TOKEN}
    public_url = ngrok.connect(8000)
    print(f"\n{'='*60}")
    print(f"  🌐  Public URL:        {public_url}")
    print(f"  💬  Chat Client:       {public_url}/client")
    print(f"  🔍  Health Check:      {public_url}/health")
    print(f"{'='*60}")

## 5. Quick API Test

Verify the server responds correctly via the tunnel.

In [ ]:
import urllib.request, json

# Detect tunnel URL
url = None

# Check cloudflared
try:
    with open("tunnel.log") as f:
        output = f.read()
    import re
    match = re.search(r'https://[a-zA-Z0-9_-]+\.trycloudflare\.com', output)
    if match:
        url = match.group(0)
except FileNotFoundError:
    pass

if not url:
    url = "http://localhost:8000"

print(f"Testing: {url}/health")
try:
    req = urllib.request.Request(f"{url}/health")
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read())
    print(json.dumps(data, indent=2))
    print("\n✅ Server is responding correctly!")
except Exception as e:
    print(f"Error: {e}")
    print("\nThe server may still be loading. Check the logs above.")

## 6. Taking Screenshots & Recordings

### Recommended Workflow

1. **Open the Chat Client URL** (from Cell 4 or 5) in your browser
2. **Send example prompts** from the list below
3. **Capture screenshots** or **record your screen** while the model responds

### Example Prompts to Demo

| Category | Prompt |
|----------|--------|
| Math | `Solve for x: 3x² + 5x - 2 = 0` |
| Code | `Write a Python function to merge two sorted lists` |
| Reasoning | `If a train leaves Station A at 60 mph and another leaves Station B at 80 mph...` |
| STEM | `Explain how a quantum computer differs from a classical computer` |

### Tips

- Use the **Settings** drawer (top-right) to adjust temperature, max tokens, and system prompt
- The **Clear** button resets the conversation session
- The model badge shows device and load status
- Response streaming shows tokens as they're generated — great for demos

### Recording Software

- **Chrome/Edge**: Use built-in "Record tab" or "Record desktop" in DevTools
- **OBS Studio**: Free, open-source screen recorder
- **Kap** (macOS): Lightweight screen recorder
- **GNOME Screenshot** (Linux): Built-in screen recorder (Ctrl+Shift+Alt+R)

## 7. Cleanup

Stop the server and tunnel when you're done.

In [ ]:
print("Stopping processes...")
!pkill -f "cloudflared" 2>/dev/null || true
!pkill -f "run.py" 2>/dev/null || true
!pkill -f "uvicorn" 2>/dev/null || true

# Close ngrok tunnels if any
try:
    from pyngrok import ngrok
    ngrok.kill()
except Exception:
    pass

print("✅ Done. Server and tunnel stopped.")